# SAM Batch Export — CoralNet Points → COCO JSON

Uses **SAM ViT-B** with **Automatic Mask Generator (AMG)** to segment all objects in each image, then assigns class labels from CoralNet point annotations via majority vote.

Produces `sam_coco.json` which `segformer_train.ipynb` (stage = `"sam_masks"`) consumes directly.

## What each cell does

1. **Setup** — install packages, download SAM checkpoint, mount Drive (conditional kernel restart only on numpy binary mismatch)
2. **Config** — paths, AMG parameters, resume flag, preview image count
3. **Load CSV + match files** — annotations indexed by image, filename resolution with fallbacks
4. **Load SAM** — GPU + optional FP16 (image-encoder-only half-precision so AMG still works)
5. **PREVIEW** — 10 images (3 seed + 7 random from dataset, seeded). Inspect before full run.
6. **Process all images** — resumable, checkpoint every 10 images, background image preloading
7. **Save COCO JSON** — final output with embedded pipeline settings
8. **Summary + quality check** — class distribution, annotations per image
9. **Compatibility check** — verify output is loadable by `segformer_train.ipynb`

## Before you start

1. Run `resize_for_colab.py` locally → get `colab_upload/` with resized images + rescaled CSV
2. Upload `colab_upload/` to `My Drive/coral_training/`
3. Runtime → Change runtime type → **L4 GPU** (~45 min for 3000 images) or **A100** (~30 min)
4. Run all cells in order

## Resume behavior

Cell 6 checkpoints every 10 images to `coral_training/sam_checkpoints/progress.json`. If Colab disconnects, just re-run Cell 6 — it picks up where it left off. Set `FORCE_FRESH = True` in Cell 2 ONLY when you want to discard progress and restart.

**Estimated time:** ~2h on T4, ~45 min on L4, ~30 min on A100 (~3000 images)

In [ ]:
# ===========================================================================
# CELL 1: Install dependencies, download SAM checkpoint, mount Drive
# ===========================================================================
# Kernel restart policy:
#   - NEVER kills on a clean run.
#   - ONLY kills if numpy is already loaded in memory with an incompatible
#     binary (the classic "dtype size changed" / "binary incompatibility"
#     error). In that case there is no other way — the stale numpy is pinned
#     in the live interpreter and must be flushed.
#   - On a fresh VM, install happens BEFORE numpy is imported, so no restart
#     is needed. If you re-run this cell after success, it is a no-op.
#
# Drive mount policy:
#   - Actively verifies `/content/drive/MyDrive` is reachable.
#   - If not yet mounted → mounts now.
#   - If already mounted and reachable → skips silently.
#   - Never prints "already mounted" unless it truly is.
# ===========================================================================

import os, subprocess, sys

_MARKER = '/content/.sam_deps_installed'

if not os.path.exists(_MARKER):
    print('Installing dependencies (one-time per VM)...')
    _pkgs = [
        "numpy>=2,<3",
        "segment-anything",
        "opencv-python-headless",
    ]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_pkgs], check=True)
    # Force numpy back to 2.x in case anything pulled it down
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "--force-reinstall", "--no-deps", "numpy>=2,<3"],
        check=True,
    )
    with open(_MARKER, 'w') as f:
        f.write('ok')
    print('✅ Dependencies installed.')
else:
    print('✅ Dependencies already installed (marker present) — skipping pip.')


# --- SAM checkpoint download (small, one-time) ---
SAM_CHECKPOINT = '/content/sam_vit_b_01ec64.pth'
if not os.path.exists(SAM_CHECKPOINT):
    print('Downloading SAM ViT-B checkpoint (~375 MB) ...')
    subprocess.run(
        ['wget', '-q',
         'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth',
         '-O', SAM_CHECKPOINT],
        check=True,
    )
    print('✅ SAM checkpoint downloaded.')
else:
    print(f'✅ SAM checkpoint already at {SAM_CHECKPOINT}')


# --- Drive mount (verify it actually mounted before skipping) ---
_DRIVE_ROOT = '/content/drive'
_DRIVE_MYDRIVE = '/content/drive/MyDrive'

def _drive_is_really_mounted():
    # MyDrive must exist AND be non-empty-listable (mount point without mount
    # exists as an empty dir, so we also check it is a FUSE mount).
    if not os.path.isdir(_DRIVE_MYDRIVE):
        return False
    try:
        # listdir will error or return [] if not mounted; a real mount has content
        # for any user with a non-empty Drive. Fall back to os.path.ismount.
        return os.path.ismount(_DRIVE_ROOT) or len(os.listdir(_DRIVE_MYDRIVE)) > 0
    except OSError:
        return False

if _drive_is_really_mounted():
    print(f'✅ Drive already mounted at {_DRIVE_ROOT} (verified).')
else:
    print('Mounting Google Drive ...')
    from google.colab import drive
    drive.mount(_DRIVE_ROOT)
    if not _drive_is_really_mounted():
        raise RuntimeError(
            f'Drive did not mount correctly at {_DRIVE_ROOT}. '
            f'Re-run this cell and accept the auth prompt.'
        )
    print(f'✅ Drive mounted at {_DRIVE_ROOT}.')


# --- Imports with numpy mismatch detection ---
def _kill_with_message(reason):
    print(f'⚠️ {reason}')
    print('   Restarting kernel automatically — click ▶ on this cell again to continue.')
    os.kill(os.getpid(), 9)

try:
    import numpy as np
    import pandas as pd
    import cv2
except ValueError as e:
    if 'dtype size changed' in str(e) or 'binary incompatibility' in str(e):
        _kill_with_message('numpy binary incompatibility — kernel has stale numpy in memory')
    raise
except ImportError as e:
    if 'numpy' in str(e).lower():
        _kill_with_message(f'Import error hinting at numpy version issue: {e}')
    raise

print(f'\nnumpy {np.__version__} — setup complete.')

In [ ]:
# ===========================================================================
# CELL 2: Configuration
# ===========================================================================


# ═══════════════════════════════════════════════════════════════════════════
# LABEL MERGE MAP — 2025-07 taxonomy simplification (94 → 49 classes)
# Embedded inline so this notebook is fully self-contained.
# Source of truth: colab/label_mapping.py in the project repo.
# ═══════════════════════════════════════════════════════════════════════════
_LABEL_MERGE_MAP = {
    # → Rock
    'TA': 'Rock', 'HS_AR': 'Rock', 'CCA': 'Rock',
    'Dead (ex)': 'Rock', 'Biofilm ex': 'Rock',
    # → SP
    'Didae': 'SP', 'Tun': 'SP',
    # → GA
    'Hal': 'GA', 'Fila (ex)': 'GA', 'Val': 'GA', 'Cau': 'GA',
    # → HC  (rare genera, each < 400 annotations)
    'Turb-HC': 'HC', 'Myc': 'HC', 'Echphy': 'HC', 'Cyph': 'HC',
    'Diplo': 'HC', 'Ser': 'HC', 'Lepta': 'HC', 'Mer': 'HC',
    'Oul': 'HC', 'Leptos': 'HC', 'Leptor': 'HC', 'Dun': 'HC',
    'Plero': 'HC', 'Bla': 'HC', 'Oxy': 'HC', 'Para': 'HC',
    'Pod': 'HC', 'Pachy': 'HC', 'Alv': 'HC', 'Sym': 'HC', 'Psa': 'HC',
    # → Frame / Xe / Other
    'Urchins ex': 'Frame', 'Tubmus': 'Xe',
    'Zoan': 'Other', 'Bivalve': 'Other', 'Bryo': 'Other',
}
_EXCLUDED_LABELS = {
    'Unknown', 'Unkn', 'Unk', 'MA', 'BA',
    '?', 'NA', 'nan', 'None', '', 'Off',
}

def apply_label_merge(label):
    """Map a CoralNet label to its merged class, or None if excluded."""
    if label in _EXCLUDED_LABELS:
        return None
    return _LABEL_MERGE_MAP.get(label, label)

def merge_dataframe_labels(df, label_col='Label'):
    """Apply merge map to df[label_col]; drops excluded rows. Returns (df, n_dropped)."""
    df = df.copy()
    df[label_col] = df[label_col].map(apply_label_merge)
    before = len(df)
    df = df[df[label_col].notna()].reset_index(drop=True)
    return df, before - len(df)

USE_LABEL_MERGE = True   # set False to use original 94-class CoralNet labels

# --- Paths ---
DRIVE_FOLDER   = '/content/drive/MyDrive/coral_training'
CSV_PATH       = f'{DRIVE_FOLDER}/annotations_coralnet.csv'
IMAGES_DIR     = f'{DRIVE_FOLDER}/images'
OUTPUT_JSON    = f'{DRIVE_FOLDER}/sam_coco.json'
CHECKPOINT_DIR = f'{DRIVE_FOLDER}/sam_checkpoints'   # crash-safe intermediate saves

# --- SAM settings ---
SAM_MODEL_TYPE = 'vit_b'
SAM_CHECKPOINT = '/content/sam_vit_b_01ec64.pth'

# --- AMG settings (match sam_viewer.py defaults) ---
AMG_POINTS_PER_SIDE  = 12      # grid density (higher = more masks, slower)
AMG_PRED_IOU_THRESH  = 0.58    # min predicted IoU to keep a mask
AMG_STABILITY_THRESH = 0.36    # min stability score
MAX_MASK_PCT         = 60.0    # drop masks covering > this % of image (background)

# --- Processing ---
SAVE_EVERY  = 10      # checkpoint every N images (resume-safe on disconnect)
USE_FP16    = True    # float16 inference: ~2x faster on GPU, masks nearly identical
FORCE_FRESH = False   # True = discard checkpoint and start over. Normally keep False.

# --- Preview (Cell 5) ---
# Seed images are the same as sam_viewer.py and segformer_train.ipynb so you
# can compare side-by-side. If fewer than N_TEST_IMAGES exist, the preview
# pads with random images from the dataset (seeded for stability).
TEST_IMAGES   = ["DSCN2121.jpg", "G0081188.jpg", "trainingJ10.jpg"]
N_TEST_IMAGES = 10
PREVIEW_SEED  = 42

import os
assert os.path.exists(CSV_PATH),   f'CSV not found: {CSV_PATH}'
assert os.path.isdir(IMAGES_DIR),  f'Images folder not found: {IMAGES_DIR}'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

if FORCE_FRESH:
    ckpt = os.path.join(CHECKPOINT_DIR, 'progress.json')
    if os.path.exists(ckpt):
        os.remove(ckpt)
        print('🗑️  FORCE_FRESH=True → deleted old checkpoint, starting from scratch')
    else:
        print('FORCE_FRESH=True but no checkpoint to delete (already fresh)')
else:
    ckpt = os.path.join(CHECKPOINT_DIR, 'progress.json')
    if os.path.exists(ckpt):
        print(f'ℹ️  Resumable checkpoint found at {ckpt} — Cell 6 will continue from it')

n_imgs = len([f for f in os.listdir(IMAGES_DIR)
              if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
print(f'\nFound {n_imgs} images in {IMAGES_DIR}')
print(f'SAM:  {SAM_MODEL_TYPE}  (FP16={USE_FP16})')
print(f'AMG:  grid {AMG_POINTS_PER_SIDE}  IoU>{AMG_PRED_IOU_THRESH}  stability>{AMG_STABILITY_THRESH}  maxmask={MAX_MASK_PCT}%')
print(f'Save every {SAVE_EVERY} images → {CHECKPOINT_DIR}')
print(f'Preview: {len(TEST_IMAGES)} seed + pad to {N_TEST_IMAGES} total')


In [ ]:
# ===========================================================================
# CELL 3: Load annotations CSV & index images
# ===========================================================================

import pandas as pd
import cv2
import os

# Load CSV (already rescaled by resize_for_colab.py)
df = pd.read_csv(CSV_PATH, low_memory=False)

# Find label column
label_col = None
for col in ['Label', 'Label code', 'label', 'label code']:
    if col in df.columns:
        label_col = col
        break
assert label_col, f'No label column found. Columns: {list(df.columns)}'
if label_col != 'Label':
    df = df.rename(columns={label_col: 'Label'})

df = df[['Name', 'Row', 'Column', 'Label']].copy()
df['Row'] = df['Row'].astype(int)
df['Column'] = df['Column'].astype(int)

# Group by image name
annotations = {}
for name, group in df.groupby('Name'):
    annotations[name] = group

# Index images on disk (flat folder from resize_for_colab.py)
image_files = {}
for f in os.listdir(IMAGES_DIR):
    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
        image_files[f] = os.path.join(IMAGES_DIR, f)
        # Also index without double extension
        base, ext = os.path.splitext(f)
        if base.lower().endswith(('.jpg', '.jpeg', '.png')):
            image_files[base] = os.path.join(IMAGES_DIR, f)

# Match: images that have both a file AND annotations
def resolve(name):
    if name in image_files:
        return image_files[name]
    base, ext = os.path.splitext(name)
    if base.lower().endswith(('.jpg', '.jpeg', '.png')):
        if base in image_files:
            return image_files[base]
    for ext in ['.jpg', '.JPG', '.jpeg', '.png']:
        candidate = name + ext
        if candidate in image_files:
            return image_files[candidate]
    return None

matched = {}
for name in annotations:
    path = resolve(name)
    if path:
        matched[name] = path



# ── Apply label merge ────────────────────────────────────────────────────
if USE_LABEL_MERGE:
    df, _dropped = merge_dataframe_labels(df, label_col='Label')
    annotations = {name: grp for name, grp in df.groupby('Name')}
    print(f'Label merge: {_dropped} excluded annotations dropped, '
          f'{df["Label"].nunique()} merged classes remain')

# Build category list from all labels
all_labels = sorted(df['Label'].unique())
cat_map = {label: idx + 1 for idx, label in enumerate(all_labels)}

print(f'CSV: {len(df)} annotations across {len(annotations)} image names')
print(f'Image files on disk: {len(image_files)}')
print(f'Matched (have both annotations + file): {len(matched)}')
print(f'Categories: {len(cat_map)}')

if len(matched) == 0:
    print('\n⚠️  No matches! Check that image filenames in CSV match files in IMAGES_DIR.')
    print(f'Sample CSV names: {list(annotations.keys())[:5]}')
    print(f'Sample file names: {list(image_files.keys())[:5]}')

# Verify coordinates match image dimensions (sanity check)
if matched:
    sample_name = list(matched.keys())[0]
    sample_img = cv2.imread(matched[sample_name])
    sh, sw = sample_img.shape[:2]
    sdf = annotations[sample_name]
    max_row, max_col = sdf['Row'].max(), sdf['Column'].max()
    if max_row > sh or max_col > sw:
        print(f'\n⚠️ WARNING: Points exceed image dimensions!')
        print(f'   Image "{sample_name}": {sw}×{sh}')
        print(f'   Max point: col={max_col}, row={max_row}')
        print(f'   Did you run resize_for_colab.py? The CSV must be rescaled too.')
    else:
        print(f'\n✅ Coordinates match image dimensions (verified on "{sample_name}")')

In [ ]:
# ===========================================================================
# CELL 4: Load SAM model
# ===========================================================================
# Note on FP16: calling sam.half() on the full model breaks SAM's
# AutomaticMaskGenerator, because prompt_encoder.forward_with_coords casts
# coords to torch.float (fp32) and then matmuls them against a half-precision
# Gaussian matrix → RuntimeError: mat1/mat2 dtype mismatch.
#
# Safe alternative (what we do here):
#   - half-precision ONLY the image encoder (the expensive part, ~95% of
#     AMG time)
#   - keep prompt_encoder + mask_decoder in fp32
#   - pre-hook: cast encoder input fp32 → fp16
#   - post-hook: cast encoder output fp16 → fp32 so the fp32 decoder is happy
# This yields nearly the full FP16 speedup without the prompt-encoder crash.
# If anything still goes wrong, set USE_FP16 = False in Cell 2.
# ===========================================================================

import torch
import numpy as np
import cv2
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

sam = sam_model_registry[SAM_MODEL_TYPE](checkpoint=SAM_CHECKPOINT)
sam.to(device)

if USE_FP16 and device == 'cuda':
    sam.image_encoder.half()

    def _cast_in_to_half(module, args):
        if not args:
            return None
        x = args[0]
        if torch.is_tensor(x) and x.dtype != torch.float16:
            return (x.half(),) + args[1:]
        return None

    def _cast_out_to_fp32(module, inp, out):
        if torch.is_tensor(out) and out.dtype != torch.float32:
            return out.float()
        return out

    sam.image_encoder.register_forward_pre_hook(_cast_in_to_half)
    sam.image_encoder.register_forward_hook(_cast_out_to_fp32)
    print('Enabled FP16 on image_encoder (prompt_encoder + mask_decoder stay fp32)')
else:
    print('FP32 inference (USE_FP16=False or no CUDA)')

print('SAM model loaded!')

In [ ]:
# ===========================================================================
# CELL 5: PREVIEW — Process N fixed/random test images and visualize masks
#          Seed images are the SAME as sam_viewer.py and segformer_train.ipynb.
#          Inspect these before running the full batch! Abort if wrong.
# ===========================================================================

import random
import colorsys
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter as _Ctr

# --- Select preview images: seed set + random pad (seeded, stable across runs) ---
_seed_avail = [n for n in TEST_IMAGES if n in matched]
_missing    = [n for n in TEST_IMAGES if n not in matched]
if _missing:
    print(f'⚠️ Missing seed test images (skipped): {_missing}')

preview_names = list(_seed_avail)
if len(preview_names) < N_TEST_IMAGES:
    rng = random.Random(PREVIEW_SEED)
    pool = [n for n in matched.keys() if n not in preview_names]
    rng.shuffle(pool)
    preview_names.extend(pool[:N_TEST_IMAGES - len(preview_names)])

preview_names = preview_names[:N_TEST_IMAGES]
if not preview_names:
    raise RuntimeError('No preview images available — check Cell 3 matching.')

print(f'✅ Preview on {len(preview_names)} images '
      f'({len(_seed_avail)} seed + {len(preview_names) - len(_seed_avail)} random):')
for i, n in enumerate(preview_names, 1):
    tag = '(seed)' if n in _seed_avail else '(random)'
    print(f'  {i:2d}. {n}  {tag}')

# --- Distinct colors per class (golden-ratio HSV — no two greens alike) ---
_GOLDEN = 0.61803398875

def _distinct_color(i):
    h = (0.12 + i * _GOLDEN) % 1.0                 # 0.12 starts off-green
    s = 0.70 + 0.25 * ((i * 7) % 3) / 2            # 0.70 / 0.825 / 0.95
    v = 0.72 + 0.25 * ((i * 11) % 2)               # 0.72 / 0.97
    r, g, b = colorsys.hsv_to_rgb(h, s, v)
    return (int(r * 255), int(g * 255), int(b * 255))

_preview_palette = {}
def _pcol(label):
    if label not in _preview_palette:
        _preview_palette[label] = _distinct_color(hash(label) & 0xFFFF)
    return _preview_palette[label]


def _put_readable_text(img, text, center_xy, fg_color):
    """Draw large, readable, high-contrast text centered at center_xy."""
    h, w = img.shape[:2]
    font       = cv2.FONT_HERSHEY_SIMPLEX
    scale      = max(0.9, min(w, h) / 1400.0)
    thick_fg   = max(2, int(round(scale * 2)))
    thick_bg   = thick_fg + 3
    (tw, th), _ = cv2.getTextSize(text, font, scale, thick_fg)
    x, y = int(center_xy[0]) - tw // 2, int(center_xy[1]) + th // 2
    x = max(3, min(w - tw - 3, x))
    y = max(th + 3, min(h - 3, y))
    cv2.putText(img, text, (x, y), font, scale, (0, 0, 0),         thick_bg, cv2.LINE_AA)
    cv2.putText(img, text, (x, y), font, scale, (255, 255, 255),   thick_fg, cv2.LINE_AA)


fig, axes = plt.subplots(len(preview_names), 2, figsize=(22, 8 * len(preview_names)))
if len(preview_names) == 1:
    axes = [axes]

for row_idx, name in enumerate(preview_names):
    img_path = matched[name]
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        print(f'⚠️ Could not read image: {img_path}')
        axes[row_idx][0].set_title(f'{name}\nFAILED TO LOAD', fontsize=11, color='red')
        axes[row_idx][0].axis('off')
        axes[row_idx][1].axis('off')
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    total_px = h * w
    max_area = total_px * MAX_MASK_PCT / 100

    mask_gen_preview = SamAutomaticMaskGenerator(
        model=sam,
        points_per_side=AMG_POINTS_PER_SIDE,
        pred_iou_thresh=AMG_PRED_IOU_THRESH,
        stability_score_thresh=AMG_STABILITY_THRESH,
        min_mask_region_area=max(100, int(total_px * 0.001)),
    )

    ann_df = annotations[name].copy()
    ann_df['Row'] = ann_df['Row'].clip(0, h - 1)
    ann_df['Column'] = ann_df['Column'].clip(0, w - 1)

    # Step 1: AMG
    amg_masks = mask_gen_preview.generate(img_rgb)
    amg_masks.sort(key=lambda m: m['area'])
    amg_filtered = [m for m in amg_masks if m['area'] <= max_area]

    # Step 2: match points to smallest containing mask
    used_mask_ids = {}
    for _, r in ann_df.iterrows():
        cx, cy = int(r['Column']), int(r['Row'])
        for mi, m in enumerate(amg_filtered):
            if m['segmentation'][cy, cx]:
                used_mask_ids.setdefault(mi, []).append((r['Label'], (cx, cy)))
                break

    # Step 3: majority vote per mask
    kept = []
    for mi, lps in used_mask_ids.items():
        m = amg_filtered[mi]
        votes = _Ctr(lp[0] for lp in lps)
        label = votes.most_common(1)[0][0]
        area = int(m['segmentation'].sum())
        kept.append({'label': label, 'mask': m['segmentation'],
                     'score': float(m['predicted_iou']), 'area_px': area})

    # Step 4: overlap resolution — highest confidence wins
    kept.sort(key=lambda s: -s['score'])
    claimed = np.zeros((h, w), dtype=bool)
    resolved = []
    for seg in kept:
        seg['mask'] = seg['mask'] & ~claimed
        area_px = int(seg['mask'].sum())
        if area_px < 50:
            continue
        claimed |= seg['mask']
        seg['area_px'] = area_px
        resolved.append(seg)
    kept = resolved

    # --- Left: original + points (with class names near each point) ---
    orig_viz = img_rgb.copy()
    for _, r in ann_df.iterrows():
        cx, cy = int(r['Column']), int(r['Row'])
        col = _pcol(r['Label'])
        cv2.circle(orig_viz, (cx, cy), 10, (0, 0, 0), -1)
        cv2.circle(orig_viz, (cx, cy), 7, col, -1)

    axes[row_idx][0].imshow(orig_viz)
    axes[row_idx][0].set_title(f'{name}\n{len(ann_df)} points, {len(amg_filtered)} AMG masks',
                                fontsize=11)
    axes[row_idx][0].axis('off')

    # --- Right: masks overlay + LARGE class name labels per region ---
    overlay = img_rgb.copy().astype(np.float32)
    for seg in kept:
        color = np.array(_pcol(seg['label']), dtype=np.float32)
        overlay[seg['mask']] = color
    result = cv2.addWeighted(img_rgb.astype(np.float32), 0.5, overlay, 0.5, 0).astype(np.uint8)

    # Contours for edges
    for seg in kept:
        col = _pcol(seg['label'])
        contours, _ = cv2.findContours(seg['mask'].astype(np.uint8),
                                        cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(result, contours, -1, (255, 255, 255), 2)
        cv2.drawContours(result, contours, -1, col, 1)

    # Large readable class labels at region centroids
    for seg in kept:
        ys, xs = np.where(seg['mask'])
        if len(xs) == 0:
            continue
        mcx, mcy = int(xs.mean()), int(ys.mean())
        # move centroid into the region if the shape is concave
        if not seg['mask'][mcy, mcx]:
            mcy, mcx = int(ys[len(ys)//2]), int(xs[len(xs)//2])
        pct = seg['area_px'] / total_px * 100
        _put_readable_text(result, f"{seg['label']}  {pct:.1f}%", (mcx, mcy), _pcol(seg['label']))

    union_pct = claimed.sum() / total_px * 100
    axes[row_idx][1].imshow(result)
    axes[row_idx][1].set_title(f'{len(kept)} segments, {union_pct:.0f}% coverage', fontsize=11)
    axes[row_idx][1].axis('off')

    import gc; torch.cuda.empty_cache(); gc.collect()

# Legend
patches = [mpatches.Patch(color=[c / 255 for c in _pcol(lbl)], label=lbl)
           for lbl in sorted(_preview_palette.keys())]
fig.legend(handles=patches, loc='lower center', ncol=min(10, len(patches)),
           fontsize=9, framealpha=0.9, bbox_to_anchor=(0.5, -0.005))
plt.suptitle(f'PREVIEW — {len(preview_names)} test images (seed + random)\n'
             f'Left: original + annotation points | Right: SAM masks with large class labels',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\n✅ Preview done. Compare with SAM Viewer webapp (🧪 button).')
print(f'⚠️ If masks look wrong, adjust AMG settings in Cell 2 and re-run.')

In [ ]:
# ===========================================================================
# CELL 6: Process all images — AMG + conflict resolution + overlap resolution
#          Logic is identical to the SAM Viewer webapp pipeline.
# ===========================================================================

import json
import time
import gc
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

# --- Background image preloader (loads next image while GPU works) ---
def load_image(img_path):
    """Load and convert image — runs in background thread."""
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        return None, None
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    return img_bgr, img_rgb

# Check for existing checkpoint to resume from
checkpoint_file = os.path.join(CHECKPOINT_DIR, 'progress.json')
if os.path.exists(checkpoint_file):
    with open(checkpoint_file) as f:
        progress = json.load(f)
    coco_images = progress['images']
    coco_annotations = progress['annotations']
    processed_names = set(progress['processed_names'])
    next_ann_id = progress['next_ann_id']
    next_img_id = progress['next_img_id']
    print(f'Resuming from checkpoint: {len(processed_names)} images already done')
else:
    coco_images = []
    coco_annotations = []
    processed_names = set()
    next_ann_id = 1
    next_img_id = 1
    print('Starting fresh')

# Stats
stats = Counter()
t0 = time.time()
total = len(matched)
remaining = [(name, path) for name, path in matched.items() if name not in processed_names]
print(f'Processing {len(remaining)} images ({len(processed_names)} already done)...\n')

def save_checkpoint():
    ckpt = {
        'images': coco_images, 'annotations': coco_annotations,
        'processed_names': list(processed_names),
        'next_ann_id': next_ann_id, 'next_img_id': next_img_id,
    }
    tmp = checkpoint_file + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(ckpt, f)
    os.replace(tmp, checkpoint_file)

# Track last min_mask_region_area to avoid recreating generator needlessly
_last_min_area = None
mask_gen = None

executor = ThreadPoolExecutor(max_workers=2)

# Pre-submit first image load
if remaining:
    next_future = executor.submit(load_image, remaining[0][1])

for idx, (name, img_path) in enumerate(remaining):
    try:
        # Get preloaded image (or load if first)
        img_bgr, img_rgb = next_future.result()

        # Pre-load NEXT image in background while GPU works on current
        if idx + 1 < len(remaining):
            next_future = executor.submit(load_image, remaining[idx + 1][1])

        if img_rgb is None:
            stats['skip_unreadable'] += 1
            processed_names.add(name)
            continue

        h, w = img_rgb.shape[:2]
        total_px = h * w
        max_area = total_px * MAX_MASK_PCT / 100

        # Create/update AMG generator (identical to webapp)
        min_region = max(100, int(total_px * 0.001))
        if mask_gen is None or min_region != _last_min_area:
            mask_gen = SamAutomaticMaskGenerator(
                model=sam,
                points_per_side=AMG_POINTS_PER_SIDE,
                pred_iou_thresh=AMG_PRED_IOU_THRESH,
                stability_score_thresh=AMG_STABILITY_THRESH,
                min_mask_region_area=min_region,
            )
            _last_min_area = min_region

        # Get annotations for this image (coordinates already rescaled by resize_for_colab.py)
        ann_df = annotations[name].copy()
        ann_df['Row'] = ann_df['Row'].clip(0, h - 1)
        ann_df['Column'] = ann_df['Column'].clip(0, w - 1)

        # --- Step 1: Generate ALL masks with AMG ---
        amg_masks = mask_gen.generate(img_rgb)
        stats['amg_masks_raw'] += len(amg_masks)

        # Sort by area ascending (smallest first) and filter large masks
        amg_masks.sort(key=lambda m: m['area'])
        amg_filtered = [m for m in amg_masks if m['area'] <= max_area]
        stats['amg_masks_filtered'] += len(amg_filtered)

        # --- Step 2: Match annotation points to smallest containing mask ---
        used_mask_ids = {}
        for _, row in ann_df.iterrows():
            cx, cy = int(row['Column']), int(row['Row'])
            label = row['Label']

            found = False
            for mi, m in enumerate(amg_filtered):
                if m['segmentation'][cy, cx]:
                    if mi not in used_mask_ids:
                        used_mask_ids[mi] = []
                    used_mask_ids[mi].append((label, (cx, cy)))
                    found = True
                    break
            if not found:
                stats['unmatched_pts'] += 1

        stats['matched_pts'] += sum(len(v) for v in used_mask_ids.values())

        # --- Step 3: Label masks via majority vote ---
        kept = []
        for mi, label_points in used_mask_ids.items():
            m = amg_filtered[mi]
            mask_bool = m['segmentation']
            area_px = int(mask_bool.sum())

            label_votes = Counter(lp[0] for lp in label_points)
            label = label_votes.most_common(1)[0][0]

            kept.append({
                'label': label, 'mask': mask_bool,
                'score': float(m['predicted_iou']),
                'area_px': area_px,
            })

        # --- Step 3b: Conflict resolution — dissenting points get own masks ---
        used_mask_set = set(used_mask_ids.keys())
        extra_segs = []
        for mi, label_points in used_mask_ids.items():
            label_votes = Counter(lp[0] for lp in label_points)
            if len(label_votes) <= 1:
                continue
            majority_label = label_votes.most_common(1)[0][0]
            for lbl, (cx, cy) in label_points:
                if lbl == majority_label:
                    continue
                best_ai = None
                best_area = float('inf')
                for ai, am in enumerate(amg_filtered):
                    if ai == mi or ai in used_mask_set:
                        continue
                    if am['area'] > max_area:
                        continue
                    if am['segmentation'][cy, cx] and am['area'] < best_area:
                        best_ai = ai
                        best_area = am['area']
                if best_ai is not None:
                    am = amg_filtered[best_ai]
                    used_mask_set.add(best_ai)
                    alt_area = int(am['segmentation'].sum())
                    extra_segs.append({
                        'label': lbl,
                        'mask': am['segmentation'],
                        'score': float(am['predicted_iou']),
                        'area_px': alt_area,
                    })
                    stats['conflict_resolved'] += 1
        if extra_segs:
            kept.extend(extra_segs)

        # --- Step 4: Resolve overlaps — highest confidence wins ---
        #   Identical to webapp: carve overlaps, drop if area < 50.
        kept.sort(key=lambda s: -s['score'])
        claimed = np.zeros((h, w), dtype=bool)
        resolved = []
        for seg in kept:
            seg['mask'] = seg['mask'] & ~claimed
            area_px = int(seg['mask'].sum())
            if area_px < 50:
                continue
            claimed |= seg['mask']
            seg['area_px'] = area_px
            resolved.append(seg)
        kept = resolved

        # --- Step 5: Extract polygons → COCO annotations ---
        img_id = next_img_id
        next_img_id += 1
        coco_images.append({
            'id': img_id, 'file_name': os.path.basename(img_path),
            'width': w, 'height': h,
        })

        n_segs = 0
        for seg in kept:
            contours, _ = cv2.findContours(
                seg['mask'].astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
            )
            min_c_area = max(50, int(total_px * 0.0001))
            polys = []
            for c in contours:
                if len(c) < 3 or cv2.contourArea(c) < min_c_area:
                    continue
                poly = c.flatten().tolist()
                if len(poly) < 6:
                    continue
                polys.append(poly)

            if polys:
                pts_all = np.concatenate([np.array(p).reshape(-1, 2) for p in polys])
                x, y, bw, bh = cv2.boundingRect(pts_all.astype(np.int32))
                coco_annotations.append({
                    'id': next_ann_id, 'image_id': img_id,
                    'category_id': cat_map[seg['label']],
                    'segmentation': polys,
                    'bbox': [int(x), int(y), int(bw), int(bh)],
                    'area': float(seg['area_px']),
                    'iscrowd': 0,
                })
                next_ann_id += 1
                n_segs += 1
            stats['labeled_masks'] += 1

        processed_names.add(name)

        # --- Progress ---
        done = len(processed_names)
        elapsed = time.time() - t0
        rate = (idx + 1) / elapsed if elapsed > 0 else 0
        eta = (len(remaining) - idx - 1) / rate / 60 if rate > 0 else 0
        if (idx + 1) % 10 == 0 or idx == 0:
            print(f'[{done}/{total}] {name} → {n_segs} polygons '
                  f'from {len(amg_filtered)} AMG | {rate:.1f} img/s | ETA {eta:.0f} min')

        # --- Checkpoint ---
        if (idx + 1) % SAVE_EVERY == 0:
            save_checkpoint()
            print(f'  💾 Checkpoint saved ({done} images)')

        # Free memory periodically
        if (idx + 1) % 20 == 0:
            torch.cuda.empty_cache()
            gc.collect()

    except Exception as e:
        print(f'⚠️ Error on {name}: {e}')
        stats['errors'] += 1
        processed_names.add(name)
        save_checkpoint()
        continue

executor.shutdown(wait=False)

# Final save
save_checkpoint()
elapsed_total = time.time() - t0
print(f'\n{"=" * 60}')
print(f'Done! Processed {len(remaining)} images in {elapsed_total/60:.1f} min')
print(f'Stats: {dict(stats)}')

In [ ]:
# ===========================================================================
# CELL 7: Build and save final COCO JSON
# ===========================================================================
# Writes sam_coco.json with embedded pipeline settings. Consumed by:
#   - segformer_train.ipynb  (set STAGE = "sam_masks")
# ===========================================================================

import json
from datetime import datetime

coco_dict = {
    'info': {
        'description': 'SAM-generated coral reef segmentation masks from CoralNet point annotations',
        'version': '2.0',
        'date_created': datetime.now().strftime('%Y-%m-%d %H:%M'),
        'pipeline_settings': {
            'sam_model': SAM_MODEL_TYPE,
            'float16': USE_FP16,
            'amg_points_per_side': AMG_POINTS_PER_SIDE,
            'amg_pred_iou_thresh': AMG_PRED_IOU_THRESH,
            'amg_stability_thresh': AMG_STABILITY_THRESH,
            'max_mask_pct': MAX_MASK_PCT,
            'overlap_resolution': 'highest_confidence_wins',
            'conflict_resolution': True,
            'label_method': 'majority_vote',
            'total_images_processed': len(coco_images),
            'total_annotations': len(coco_annotations),
            'processing_stats': dict(stats),
        },
    },
    'licenses': [{'id': 1, 'name': 'Unknown', 'url': ''}],
    'categories': [
        {'id': cid, 'name': name, 'supercategory': 'coral'}
        for name, cid in sorted(cat_map.items(), key=lambda x: x[1])
    ],
    'images': coco_images,
    'annotations': coco_annotations,
}

with open(OUTPUT_JSON, 'w') as f:
    json.dump(coco_dict, f)

size_mb = os.path.getsize(OUTPUT_JSON) / 1024 / 1024
print(f'Saved: {OUTPUT_JSON}  ({size_mb:.1f} MB)')
print(f'  Images:      {len(coco_images)}')
print(f'  Annotations: {len(coco_annotations)}')
print(f'  Categories:  {len(cat_map)}')

if len(processed_names) == len(matched):
    if os.path.exists(checkpoint_file):
        os.remove(checkpoint_file)
        print('\n🧹 Checkpoint cleaned up (processing complete).')

print('\n✅ Ready for segformer_train.ipynb — set STAGE = "sam_masks" in Cell 2.')

In [ ]:
# ===========================================================================
# CELL 8: Summary stats & quality check
# ===========================================================================

from collections import Counter
import matplotlib.pyplot as plt

class_counts = Counter()
for ann in coco_annotations:
    cid = ann['category_id']
    name = [c['name'] for c in coco_dict['categories'] if c['id'] == cid][0]
    class_counts[name] += 1

anns_per_img = Counter()
for ann in coco_annotations:
    anns_per_img[ann['image_id']] += 1

print('=== Output Dataset Summary ===')
print(f'Images:       {len(coco_images)}')
print(f'Annotations:  {len(coco_annotations)}')
print(f'Categories:   {len(cat_map)}')
print(f'Avg ann/img:  {len(coco_annotations) / max(len(coco_images), 1):.1f}')
if anns_per_img:
    print(f'Max ann/img:  {max(anns_per_img.values())}')
    print(f'Min ann/img:  {min(anns_per_img.values())}')

print('\nTop 20 classes:')
for cls, count in class_counts.most_common(20):
    print(f'  {cls:<25} {count:>5}')

print('\nBottom 10 classes (consider dropping if count < 10):')
for cls, count in class_counts.most_common()[-10:]:
    print(f'  {cls:<25} {count:>5}')

fig, ax = plt.subplots(figsize=(14, 6))
top30 = class_counts.most_common(30)
ax.barh([c[0] for c in top30][::-1], [c[1] for c in top30][::-1])
ax.set_xlabel('Annotations')
ax.set_title('Top 30 classes by annotation count (SAM COCO output)')
plt.tight_layout()
plt.show()

# ===========================================================================
# CELL 9: Compatibility check — verify sam_coco.json works with segformer_train.ipynb
# ===========================================================================
# segformer_train.ipynb's Cell 5 (render_coco_to_png) maps COCO category names
# to its own class index. If names don't match exactly, those annotations are
# silently dropped. This cell cross-checks before you start training.
# ===========================================================================

import json
import pandas as pd

with open(OUTPUT_JSON) as f:
    coco_loaded = json.load(f)

# Derive segformer's class list the same way segformer_train.ipynb does
df_check = pd.read_csv(CSV_PATH, low_memory=False)
seg_label_col = 'Label code' if 'Label code' in df_check.columns else 'Label'
segformer_classes = set(df_check[seg_label_col].astype(str).unique())

coco_categories = set(c['name'] for c in coco_loaded['categories'])

missing_in_coco     = segformer_classes - coco_categories
missing_in_segformer = coco_categories - segformer_classes

print(f'Classes in CSV  (segformer will build these): {len(segformer_classes)}')
print(f'Classes in COCO (from this batch export):     {len(coco_categories)}')
print(f'Overlap: {len(segformer_classes & coco_categories)}')

if missing_in_coco:
    print(f'\nℹ️ In CSV but NOT in COCO ({len(missing_in_coco)}):')
    print('   (these classes had no matching SAM masks — they stay "ignore" in training)')
    for c in sorted(missing_in_coco)[:15]:
        print(f'   - {c}')
    if len(missing_in_coco) > 15:
        print(f'   ... and {len(missing_in_coco) - 15} more')

if missing_in_segformer:
    print(f'\n⚠️ In COCO but NOT in CSV ({len(missing_in_segformer)}):')
    print('   (these will be silently dropped during training! Check for whitespace/case differences)')
    for c in sorted(missing_in_segformer):
        print(f'   - "{c}"')

# Sanity: check that at least one image in COCO exists on disk
import os
sample_missing = 0
for im in coco_loaded['images'][:50]:
    if not os.path.exists(os.path.join(IMAGES_DIR, im['file_name'])):
        sample_missing += 1
if sample_missing:
    print(f'\n⚠️ {sample_missing}/50 sampled COCO image filenames not found in {IMAGES_DIR}')
else:
    print(f'\n✅ Sampled COCO image filenames all exist on disk')

# Check: annotation counts look sane
n_anns = len(coco_loaded['annotations'])
n_imgs = len(coco_loaded['images'])
if n_imgs and n_anns / n_imgs < 3:
    print(f'\n⚠️ Only {n_anns/n_imgs:.1f} anns/image on average — AMG may be too strict. '
          f'Consider lowering AMG_PRED_IOU_THRESH or AMG_STABILITY_THRESH in Cell 2.')

print('\n✅ Compatibility check complete.')
print('   Next: open segformer_train.ipynb, set STAGE = "sam_masks", run all cells.')